# Smoking is bad

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
data_path = Path("data") / "smoking_data.csv"
df = pd.read_csv(data_path, index_col=0)
df.head()

In [ ]:
# 1314 rows and 5 columns
df.shape

In [ ]:
# By default, .describe() leaves out the string columns (outcome, smoker, gender).
# With include="str" we get their count, number of unique values, most common value and its frequency.
display(df.describe())
df.describe(include="str")

In [ ]:
# Only salary has missing values: 700 out of 1314 rows (about 53%).
# Missing salaries look unrelated to smoking and survival, so we keep these rows
# and only leave them out when analysing salary itself.
df.isna().sum()

In [ ]:
# Turn the text columns into 1/0, so that the mean of a column is a proportion
df["alive"] = (df["outcome"] == "Alive").astype(int)
df["smokes"] = (df["smoker"] == "Yes").astype(int)
df.head()

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
df.groupby(["smokes"]).agg(prob=("alive", np.mean))

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

sns.boxplot(data=df, x="outcome", y="age", color="#2a78d6", ax=axes[0])
axes[0].set_title("People who died were much older")
axes[0].set_xlabel("Outcome")
axes[0].set_ylabel("Age")

sns.boxplot(data=df, x="smoker", y="age", color="#2a78d6", ax=axes[1])
axes[1].set_title("Smokers were younger on average")
axes[1].set_xlabel("Smoker")

fig.tight_layout()
plt.show()

In [ ]:
age_bins = [18, 25, 35, 45, 55, 65, 75, 85]
age_labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)

survival_by_age = df.groupby(["age_group", "smoker"], observed=True)["alive"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(
    data=survival_by_age, x="age_group", y="alive", hue="smoker",
    hue_order=["No", "Yes"], palette={"No": "#2a78d6", "Yes": "#eb6834"},
    style="smoker", markers=True, dashes=False, markersize=8, linewidth=2, ax=ax,
)
ax.set_title("Within most age groups, smokers survive less often")
ax.set_xlabel("Age group")
ax.set_ylabel("Share alive")
ax.legend(title="Smoker")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

sns.barplot(
    data=df, x="gender", y="alive", hue="smoker", hue_order=["No", "Yes"],
    palette={"No": "#2a78d6", "Yes": "#eb6834"}, errorbar=None, gap=0.1, ax=axes[0],
)
axes[0].set_title("Smokers survive more often in both genders")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Share alive")
axes[0].set_ylim(0, 1)
axes[0].legend(title="Smoker", ncols=2, loc="upper center")

sns.boxplot(data=df, x="outcome", y="salary", color="#2a78d6", ax=axes[1])
axes[1].set_title("Salary is about the same for both outcomes")
axes[1].set_xlabel("Outcome")
axes[1].set_ylabel("Salary")

fig.tight_layout()
plt.show()

In [ ]:
# Age-adjusted survival: give both groups the age structure of the full sample
survival_table = df.pivot_table(
    index="age_group", columns="smoker", values="alive", aggfunc="mean", observed=True
)
age_weights = df["age_group"].value_counts(normalize=True)
survival_table.mul(age_weights, axis=0).sum()

## Conclusion

**The cousin's numbers are correct, but her interpretation is not.** Smokers do have a higher overall survival rate (0.761 vs 0.686), but this is mainly explained by age.

- **Age drives survival:** the median age is 67 among those who died and 38 among those still alive.
- **Smokers are younger:** the mean age is 44.7 for smokers and 48.7 for non-smokers. Among people aged 65 or over, there are 193 non-smokers but only 50 smokers, so the non-smoker group contains many more people with low survival rates.
- **Within age groups the result reverses:** non-smokers survive more often in 5 of the 7 age groups, and the other 2 are effectively tied. This is an example of Simpson's paradox.
- **Adjusting for age flips the overall comparison:** if both groups had the same age structure as the full sample, survival would be about 0.740 for non-smokers and 0.696 for smokers.
- **Gender does not explain the result:** smokers have the higher raw survival rate for both women and men. The gap is larger for men because male non-smokers are about 7 years older than male smokers on average.
- **Salary shows no clear link with survival:** the salary distributions are very similar for both outcomes, although salary is missing for 700 of the 1314 people.

**Limitations:** this is observational data, so it shows that the cousin's conclusion does not hold, but it does not measure the causal effect of smoking. The oldest smoker groups are also small (37 people aged 65-74 and 13 aged 75-84).

*Figures are not stored in this notebook because the nbstripout hook removes outputs on commit. Run all cells to reproduce them.*